# ISPU Forecasting Pipeline

This notebook implements the end-to-end forecasting pipeline using the **Darts** library.
It uses **LightGBM** as the core forecasting model and applies the **Permen LHK No. 14 Tahun 2020** (Tuned) logic for ISPU calculation.

In [1]:
import pandas as pd
import numpy as np
import warnings
import torch
import sys
import os

# Add current directory to path to ensure local imports work
sys.path.append(os.getcwd())

import ispu_calculator
from ispu_calculator import calculate_ispu_for_dataframe, map_to_3_categories

from darts import TimeSeries
from darts.models import LightGBMModel

warnings.filterwarnings('ignore')

/mnt/data/lomba/arkavidia itb/kerjain/.venv/lib/python3.14/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The StatsForecast module could not be imported. To enable support for the AutoARIMA, AutoETS and Croston models, please consider installing it.


In [2]:
# --- Configuration ---
POLLUTANTS = ['pm10', 'pm25', 'so2', 'co', 'o3', 'no2']
WEATHER_FEATURES = [
    'temperature_2m_max', 'temperature_2m_min', 'temperature_2m_mean',
    'precipitation_sum', 'wind_speed_10m_max', 'wind_speed_10m_mean',
    'relative_humidity_2m_mean', 'cloud_cover_mean', 'surface_pressure_mean'
]
STATIONS = ['DKI1', 'DKI2', 'DKI3', 'DKI4', 'DKI5']

# Model Params
INPUT_CHUNK = 21
OUTPUT_CHUNK = 14
FORECAST_HORIZON = 91

In [3]:
def create_series_per_station(df, value_cols, station):
    """Creates a Darts TimeSeries for a specific station."""
    df_st = df[df['stasiun'] == station].copy()
    df_st = df_st.sort_values('tanggal')
    
    # Handle duplicate dates by taking mean
    df_st = df_st.groupby('tanggal')[value_cols].mean()
    
    # Resample to daily frequency and fill missing values
    df_st = df_st.asfreq('D')
    df_st = df_st.ffill().bfill()
    return TimeSeries.from_dataframe(df_st, fill_missing_dates=True, freq='D')

## 1. Load Data

In [8]:
try:
    # Adjust paths as needed based on where the notebook is running
    df_train = pd.read_csv('../dataset/final_data_full_imputed_v3.csv', parse_dates=['tanggal'])
    # df_train = pd.read_csv('../feature/final_feature.csv', parse_dates=['tanggal'])
    df_forecast = pd.read_csv('../feature/forecast_features_enhanced_sep_nov_2025.csv', parse_dates=['tanggal'])
    
    df_train = df_train[df_train['stasiun'].isin(STATIONS)].copy()
    df_train = df_train.dropna(subset=POLLUTANTS)
    print(f"Train data shape: {df_train.shape}")
except FileNotFoundError:
    print("Error: Data files not found. Please check paths.")

Train data shape: (15410, 86)


## 2. Prepare Time Series

In [9]:
target_series = {}
cov_series = {}
future_cov_series = {}

for st in STATIONS:
    target_series[st] = create_series_per_station(df_train, POLLUTANTS, st)
    cov_series[st] = create_series_per_station(df_train, WEATHER_FEATURES, st)
    future_cov_series[st] = create_series_per_station(df_forecast, WEATHER_FEATURES, st)

full_train_list = [target_series[st] for st in STATIONS]
full_cov_list = []
for st in STATIONS:
    combined = cov_series[st].append(future_cov_series[st])
    full_cov_list.append(combined)

## 3. Train Model & Validation

In [10]:
from darts.models import RNNModel
from darts.metrics import mae, rmse
import torch

VAL_SIZE = 30
train_list = [ts[:-VAL_SIZE] for ts in full_train_list]
val_list = [ts[-VAL_SIZE:] for ts in full_train_list]
# Covariates need to extend through validation period into future forecasts
cov_train = full_cov_list

In [ ]:
print("Training models...")

lgb_model = LightGBMModel(
    lags=INPUT_CHUNK,
    lags_future_covariates=(INPUT_CHUNK, OUTPUT_CHUNK),
    output_chunk_length=OUTPUT_CHUNK,
    verbose=-1
)
lgb_model.fit(series=train_list, future_covariates=cov_train)
lgb_preds = lgb_model.predict(n=VAL_SIZE, series=train_list, future_covariates=cov_train)


Training models...


In [ ]:
lgb_mae = np.mean([mae(val_list[i], lgb_preds[i]) for i in range(len(STATIONS))])
lgb_rmse = np.mean([rmse(val_list[i], lgb_preds[i]) for i in range(len(STATIONS))])

model = lgb_model
model.fit(series=full_train_list, future_covariates=full_cov_list)

LightGBMModel(lags=21, lags_past_covariates=None, lags_future_covariates=(21, 14), output_chunk_length=14, output_chunk_shift=0, add_encoders=None, likelihood=None, quantiles=None, random_state=None, multi_models=True, use_static_covariates=True, categorical_past_covariates=None, categorical_future_covariates=None, categorical_static_covariates=None, verbose=-1)

## 4. Forecasting

In [16]:
print("Generating forecasts...")
final_preds = model.predict(n=FORECAST_HORIZON, series=full_train_list, future_covariates=full_cov_list)

Generating forecasts...


## 5. Generate Submission & ISPU Calculation

In [17]:
submissions = []
pollutant_predictions = []

for i, st in enumerate(STATIONS):
    pred_df = pd.DataFrame(
        final_preds[i].values(), 
        index=final_preds[i].time_index,
        columns=final_preds[i].components
    ).clip(lower=0)
    
    pollutant_pred_df = pred_df.copy()
    pollutant_pred_df['stasiun'] = st
    pollutant_pred_df['tanggal'] = pollutant_pred_df.index
    pollutant_predictions.append(pollutant_pred_df)
    
    pred_df = calculate_ispu_for_dataframe(pred_df, pollutant_cols=POLLUTANTS)
    pred_df['category_3class'] = pred_df['category'].apply(map_to_3_categories)
    pred_df['id'] = pred_df.index.strftime('%Y-%m-%d') + '_' + st
    pred_df['tanggal'] = pred_df.index
    
    submissions.append(pred_df[['id', 'tanggal', 'category', 'category_3class', 'max_ispu', 'critical_parameter']])

submission_df = pd.concat(submissions, ignore_index=True)
submission_df = submission_df.sort_values(['tanggal', 'id']).drop(columns=['tanggal']).reset_index(drop=True)

sample_sub = pd.read_csv('../dataset/sample_submission.csv')
submission_df = sample_sub[['id']].merge(submission_df, on='id', how='left')

submission_df[['id', 'category']].to_csv('submission.csv', index=False)
submission_df[['id', 'category_3class']].rename(columns={'category_3class': 'category'}).to_csv('submission_3class.csv', index=False)

pollutant_predictions_df = pd.concat(pollutant_predictions, ignore_index=True)
pollutant_predictions_df = pollutant_predictions_df[['tanggal', 'stasiun'] + POLLUTANTS].sort_values(['tanggal', 'stasiun']).reset_index(drop=True)
pollutant_predictions_df.to_csv('predictions_pollutants.csv', index=False)

print(f"✅ Submission: {len(submission_df)} rows | {submission_df['category'].value_counts().to_dict()}")

✅ Submission: 455 rows | {'TIDAK SEHAT': 442, 'SEDANG': 9, 'BAIK': 4}
